# Project 1 — EMG Gesture Classification (Ninapro DB2)

Foundation project for the myoelectric-controller repo (chains into [Project 4: temporal decision aggregation](../myoelectric-controller/02_temporal_decision_aggregation.ipynb) and [Project 5: quantization](../myoelectric-controller/03_quantization_efficiency.ipynb)).

**Dataset**: Ninapro DB2 — 40 healthy subjects, 49 movements, 12 Delsys Trigno wireless electrodes (forearm flexors/extensors + biceps/triceps), 2 kHz sampling, 6 repetitions per movement (5 s contraction + 3 s rest). Registration required at http://ninapro.hevs.ch/.

We use the `restimulus` / `rerepetition` fields, **not** `stimulus`/`repetition` — the `re*` versions are relabeled against glove-derived movement onset and correct for the fixed-delay stimulus cue. Most public notebooks train on the raw `stimulus` field and silently learn on mislabeled transition frames.

We subset to **Exercise B** (17 movements) and further down to an 8–10 class + rest subset — 49-way classification is a research problem, not a portfolio project.

## Pipeline
1. Load `.mat` files per subject, extract `emg`, `restimulus`, `rerepetition`.
2. Preprocess: 4th-order Butterworth bandpass 20–450 Hz, 50 Hz notch (EU recording, not 60 Hz), per-channel z-score fit on training-fold statistics only.
3. Window: 200 ms analysis window, 100 ms increment (justified against the ~300 ms controller-response budget from Englehart & Hudgins).
4. Features: MAV, waveform length, zero crossings, slope sign changes, RMS, Willison amplitude, 4th-order AR coefficients — ~12 channels × 10 features.
5. Models, escalating: LDA → linear SVM → LightGBM → 1D CNN on raw windows.
6. Three split protocols, all reported:
   - (a) **Within-subject**, standard Ninapro protocol: reps {1,3,4,6} train, {2,5} test.
   - (b) **Leave-one-subject-out (LOSO)**.
   - (c) **Random window shuffle** — *intentionally wrong*, included to quantify leakage. Expect it to inflate accuracy by 20+ points; that gap is the headline result of this notebook.

Expected honest numbers: within-subject 75–90% macro F1 on 10 classes; LOSO drops to 40–60%.


## 1. Setup

In [ ]:
# Colab setup — uncomment on first run
# !pip install -q scipy scikit-learn lightgbm torch mne-features numpy pandas matplotlib

import os
import glob
import json
import itertools
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.signal import butter, sosfiltfilt, iirnotch, tf2sos
import matplotlib.pyplot as plt

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("lightgbm not installed — pip install lightgbm to enable that arm")

RNG_SEED = 0
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


## 2. Config

In [ ]:
# Google Drive mount (Colab). Skip if running locally with DATA_DIR already populated.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = "/content/drive/MyDrive/bhuvan research project/ninapro_db2"
except ImportError:
    DATA_DIR = "./data/ninapro_db2"

@dataclass
class Config:
    data_dir: str = DATA_DIR
    fs: int = 2000                     # Hz, Ninapro DB2 sampling rate
    n_channels: int = 12
    bandpass_lo: float = 20.0
    bandpass_hi: float = 450.0
    notch_freq: float = 50.0           # EU mains, not 60 Hz
    window_ms: float = 200.0
    increment_ms: float = 100.0
    exercise: str = "B"                # Exercise B: 17 movements
    n_classes_subset: int = 9          # + rest = 10 classes
    train_reps: tuple = (1, 3, 4, 6)
    test_reps: tuple = (2, 5)
    ar_order: int = 4

CFG = Config()

@property
def window_len(self):
    return int(self.fs * self.window_ms / 1000)

@property
def increment_len(self):
    return int(self.fs * self.increment_ms / 1000)

Config.window_len = window_len
Config.increment_len = increment_len
print(CFG)


## 3. Data loading

Ninapro DB2 ships one `.mat` file per subject per exercise, e.g. `S1_E2_A1.mat`, containing `emg`, `restimulus`, `rerepetition`, `stimulus`, `repetition`.

In [ ]:
# Ninapro's .mat files are named by numeric exercise index (E1/E2/E3), not by letter --
# E1 = Exercise A, E2 = Exercise B, E3 = Exercise C. cfg.exercise stays a letter ("B") because
# that's how the exercise is described in every Ninapro paper/readme; this map is what turns
# it into the file-name-compatible number. (An earlier version of this function globbed for
# "S*_EB_A1.mat" directly, which never matches a real file -- caught while adapting this
# pipeline to run against DB5, see myoelectric-controller/run_project1_lean.py.)
EXERCISE_LETTER_TO_FILE_NUM = {"A": 1, "B": 2, "C": 3}


def list_subject_files(cfg: Config, exercise: str = None):
    exercise = exercise or cfg.exercise
    exercise_num = EXERCISE_LETTER_TO_FILE_NUM[exercise]
    pattern = os.path.join(cfg.data_dir, f"S*_E{exercise_num}_A1.mat")
    files = sorted(glob.glob(pattern))
    if not files:
        print(f"No files found under {cfg.data_dir} matching S*_E{exercise_num}_A1.mat — "
              f"download Ninapro DB2 (registration required) and set cfg.data_dir.")
    return files


def load_subject_mat(path: str) -> dict:
    m = sio.loadmat(path)
    emg = m["emg"].astype(np.float32)                    # (T, 12)
    restimulus = m["restimulus"].astype(np.int64).ravel() # relabeled ground truth
    rerepetition = m["rerepetition"].astype(np.int64).ravel()
    return {"emg": emg, "restimulus": restimulus, "rerepetition": rerepetition,
            "subject": os.path.basename(path)}


def subset_classes(rec: dict, n_classes: int, rng: np.random.RandomState = None) -> dict:
    """Keep rest (label 0) + the n_classes most frequent non-rest movements,
    then remap labels to a contiguous 0..n_classes range."""
    labels, counts = np.unique(rec["restimulus"], return_counts=True)
    non_rest = [(l, c) for l, c in zip(labels, counts) if l != 0]
    non_rest.sort(key=lambda lc: -lc[1])
    keep = [0] + [l for l, _ in non_rest[:n_classes]]
    mask = np.isin(rec["restimulus"], keep)
    remap = {old: new for new, old in enumerate(keep)}
    out = dict(rec)
    out["emg"] = rec["emg"][mask]
    out["restimulus"] = np.array([remap[v] for v in rec["restimulus"][mask]])
    out["rerepetition"] = rec["rerepetition"][mask]
    out["class_map"] = keep  # index -> original Ninapro movement id
    return out


## 4. Preprocessing

In [ ]:
def bandpass_sos(fs, lo, hi, order=4):
    return butter(order, [lo, hi], btype="bandpass", fs=fs, output="sos")


def notch_sos(fs, freq, q=30.0):
    b, a = iirnotch(freq, q, fs)
    return tf2sos(b, a)


def filter_emg(emg: np.ndarray, cfg: Config) -> np.ndarray:
    """emg: (T, C). Zero-phase bandpass + notch, causal-safe for offline analysis."""
    bp = bandpass_sos(cfg.fs, cfg.bandpass_lo, cfg.bandpass_hi)
    nt = notch_sos(cfg.fs, cfg.notch_freq)
    out = sosfiltfilt(bp, emg, axis=0)
    out = sosfiltfilt(nt, out, axis=0)
    return out.astype(np.float32)


def fit_zscore(emg_train: np.ndarray):
    """Per-channel mean/std fit ONLY on training-fold data — leakage guard."""
    mu = emg_train.mean(axis=0, keepdims=True)
    sd = emg_train.std(axis=0, keepdims=True) + 1e-8
    return mu, sd


def apply_zscore(emg, mu, sd):
    return (emg - mu) / sd


## 5. Windowing

In [ ]:
def make_windows(emg, labels, repetition, cfg: Config):
    """Slide a window_len window at increment_len hop. A window's label/repetition
    are taken from its center sample; windows straddling a repetition boundary
    (label changes inside the window) are dropped."""
    win, inc = cfg.window_len, cfg.increment_len
    n = emg.shape[0]
    starts = np.arange(0, n - win + 1, inc)
    X, y, rep = [], [], []
    for s in starts:
        e = s + win
        seg_labels = labels[s:e]
        if seg_labels[0] != seg_labels[-1]:
            continue  # boundary-straddling window, drop
        X.append(emg[s:e])
        y.append(seg_labels[win // 2])
        rep.append(repetition[s:e][win // 2])
    return np.stack(X), np.array(y), np.array(rep)


## 6. Feature extraction

Hudgins time-domain set + 4th-order AR coefficients, per channel.

In [ ]:
def feat_mav(x):        return np.mean(np.abs(x), axis=0)
def feat_wl(x):          return np.sum(np.abs(np.diff(x, axis=0)), axis=0)
def feat_zc(x, thresh=1e-3):
    sign_change = (x[:-1] * x[1:]) < 0
    big_enough = np.abs(x[:-1] - x[1:]) > thresh
    return np.sum(sign_change & big_enough, axis=0)
def feat_ssc(x, thresh=1e-3):
    d1 = np.diff(x[:-1], axis=0)
    d2 = np.diff(x[1:], axis=0)
    return np.sum(((d1 * d2) < 0) & (np.abs(d1 - d2) > thresh), axis=0)
def feat_rms(x):         return np.sqrt(np.mean(x ** 2, axis=0))
def feat_wamp(x, thresh=1e-2):
    return np.sum(np.abs(np.diff(x, axis=0)) > thresh, axis=0)

def feat_ar(x, order=4):
    """Per-channel AR coefficients via Yule-Walker (biased autocovariance)."""
    C, out = x.shape[1], []
    for c in range(C):
        sig = x[:, c] - x[:, c].mean()
        r = np.correlate(sig, sig, mode="full")[len(sig) - 1:]
        r = r[: order + 1] / len(sig)
        R = np.array([[r[abs(i - j)] for j in range(order)] for i in range(order)])
        rhs = r[1: order + 1]
        try:
            coeffs = np.linalg.solve(R + 1e-8 * np.eye(order), rhs)
        except np.linalg.LinAlgError:
            coeffs = np.zeros(order)
        out.append(coeffs)
    return np.concatenate(out)


def extract_features(window: np.ndarray, ar_order=4) -> np.ndarray:
    """window: (T, C) -> flat feature vector, ~C * (6 + ar_order)."""
    feats = [feat_mav(window), feat_wl(window), feat_zc(window),
              feat_ssc(window), feat_rms(window), feat_wamp(window)]
    flat = np.concatenate(feats)
    ar = feat_ar(window, order=ar_order)
    return np.concatenate([flat, ar])


def extract_feature_matrix(X_windows: np.ndarray, ar_order=4) -> np.ndarray:
    return np.stack([extract_features(w, ar_order) for w in X_windows])


## 7. Split protocols

In [ ]:
def split_within_subject(y, rep, cfg: Config):
    train_mask = np.isin(rep, cfg.train_reps)
    test_mask = np.isin(rep, cfg.test_reps)
    return train_mask, test_mask


def split_random_shuffle(y, rep, cfg: Config, test_frac=0.3, rng=None):
    """WRONG on purpose: ignores repetition structure, shuffles windows randomly.
    Adjacent overlapping windows from the same contraction leak across train/test."""
    rng = rng or np.random.RandomState(RNG_SEED)
    n = len(y)
    idx = rng.permutation(n)
    n_test = int(n * test_frac)
    test_idx = set(idx[:n_test])
    test_mask = np.array([i in test_idx for i in range(n)])
    return ~test_mask, test_mask


def loso_splits(subject_records: list):
    """Yield (train_subjects, test_subject) for leave-one-subject-out."""
    subjects = [r["subject"] for r in subject_records]
    for i, held_out in enumerate(subjects):
        train = [r for j, r in enumerate(subject_records) if j != i]
        test = subject_records[i]
        yield train, test


## 8. Models

In [ ]:
def fit_lda(X_train, y_train):
    clf = LinearDiscriminantAnalysis()
    clf.fit(X_train, y_train)
    return clf


def fit_linear_svm(X_train, y_train):
    clf = LinearSVC(C=1.0, max_iter=5000)
    clf.fit(X_train, y_train)
    return clf


def fit_lightgbm(X_train, y_train, n_classes):
    if not HAS_LGBM:
        raise RuntimeError("lightgbm not installed")
    clf = LGBMClassifier(n_estimators=300, num_leaves=31, objective="multiclass",
                          num_class=n_classes, random_state=RNG_SEED, verbosity=-1)
    clf.fit(X_train, y_train)
    return clf


class EMG1DCNN(nn.Module):
    """1D CNN over raw (C, T) windows."""
    def __init__(self, n_channels, n_classes, window_len):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Linear(64, n_classes)

    def forward(self, x):
        # x: (B, C, T)
        z = self.net(x).squeeze(-1)
        return self.head(z)


class WindowDataset(Dataset):
    def __init__(self, X_windows, y):
        # X_windows: (N, T, C) -> store as (N, C, T) for conv1d
        self.X = torch.tensor(X_windows, dtype=torch.float32).permute(0, 2, 1)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


def fit_cnn(X_train_windows, y_train, n_classes, cfg: Config, epochs=15, batch_size=64, lr=1e-3):
    model = EMG1DCNN(cfg.n_channels, n_classes, cfg.window_len).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(WindowDataset(X_train_windows, y_train), batch_size=batch_size, shuffle=True)
    model.train()
    for ep in range(epochs):
        tot_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            opt.step()
            tot_loss += loss.item() * len(yb)
        if (ep + 1) % 5 == 0 or ep == epochs - 1:
            print(f"  epoch {ep+1}/{epochs}  loss={tot_loss/len(y_train):.4f}")
    return model


@torch.no_grad()
def predict_cnn(model, X_windows, batch_size=256):
    model.eval()
    loader = DataLoader(WindowDataset(X_windows, np.zeros(len(X_windows))), batch_size=batch_size)
    preds = []
    for xb, _ in loader:
        out = model(xb.to(DEVICE))
        preds.append(out.argmax(dim=1).cpu().numpy())
    return np.concatenate(preds)


## 9. End-to-end run: load → preprocess → window → feature/model per protocol

In [ ]:
def prepare_subject(path, cfg: Config):
    rec = load_subject_mat(path)
    rec = subset_classes(rec, cfg.n_classes_subset)
    rec["emg"] = filter_emg(rec["emg"], cfg)
    return rec


def evaluate_split(X_train_w, y_train, X_test_w, y_test, n_classes, cfg: Config, model_names=("lda", "svm", "lgbm", "cnn")):
    """Feature-based models share one feature matrix; z-score fit on train only."""
    results = {}

    # feature-domain
    F_train = extract_feature_matrix(X_train_w, cfg.ar_order)
    F_test = extract_feature_matrix(X_test_w, cfg.ar_order)
    scaler = StandardScaler().fit(F_train)
    F_train_s, F_test_s = scaler.transform(F_train), scaler.transform(F_test)

    if "lda" in model_names:
        clf = fit_lda(F_train_s, y_train)
        pred = clf.predict(F_test_s)
        results["lda"] = score(y_test, pred)

    if "svm" in model_names:
        clf = fit_linear_svm(F_train_s, y_train)
        pred = clf.predict(F_test_s)
        results["svm"] = score(y_test, pred)

    if "lgbm" in model_names and HAS_LGBM:
        clf = fit_lightgbm(F_train_s, y_train, n_classes)
        pred = clf.predict(F_test_s)
        results["lgbm"] = score(y_test, pred)

    if "cnn" in model_names:
        # per-channel z-score on raw windows, train stats only
        mu = X_train_w.mean(axis=(0, 1), keepdims=True)
        sd = X_train_w.std(axis=(0, 1), keepdims=True) + 1e-8
        Xtr = (X_train_w - mu) / sd
        Xte = (X_test_w - mu) / sd
        model = fit_cnn(Xtr, y_train, n_classes, cfg)
        pred = predict_cnn(model, Xte)
        results["cnn"] = score(y_test, pred)

    return results


def score(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro")}


In [ ]:
files = list_subject_files(CFG)
all_results = []

if files:
    subjects = [prepare_subject(f, CFG) for f in files]
    n_classes = CFG.n_classes_subset + 1

    for rec in subjects:
        emg, y, rep_full = rec["emg"], rec["restimulus"], rec["rerepetition"]
        Xw, yw, repw = make_windows(emg, y, rep_full, CFG)

        # (a) within-subject
        tr_mask, te_mask = split_within_subject(yw, repw, CFG)
        if tr_mask.sum() > 0 and te_mask.sum() > 0:
            res = evaluate_split(Xw[tr_mask], yw[tr_mask], Xw[te_mask], yw[te_mask], n_classes, CFG)
            for model, m in res.items():
                all_results.append({"subject": rec["subject"], "protocol": "within_subject",
                                     "model": model, **m})

        # (c) random shuffle — intentionally leaky, for comparison
        tr_mask, te_mask = split_random_shuffle(yw, repw, CFG)
        res = evaluate_split(Xw[tr_mask], yw[tr_mask], Xw[te_mask], yw[te_mask], n_classes, CFG)
        for model, m in res.items():
            all_results.append({"subject": rec["subject"], "protocol": "random_shuffle_leaky",
                                 "model": model, **m})

    # (b) LOSO — pool windows per subject once, then hold one out at a time
    windowed = []
    for rec in subjects:
        Xw, yw, repw = make_windows(rec["emg"], rec["restimulus"], rec["rerepetition"], CFG)
        windowed.append({"subject": rec["subject"], "X": Xw, "y": yw})

    for train_recs, test_rec in loso_splits(windowed):
        X_train = np.concatenate([r["X"] for r in train_recs])
        y_train = np.concatenate([r["y"] for r in train_recs])
        res = evaluate_split(X_train, y_train, test_rec["X"], test_rec["y"], n_classes, CFG,
                              model_names=("lda", "svm"))  # cheaper arms for LOSO sweep
        for model, m in res.items():
            all_results.append({"subject": test_rec["subject"], "protocol": "loso",
                                 "model": model, **m})

    results_df = pd.DataFrame(all_results)
    results_df.to_csv("project1_results.csv", index=False)
    display(results_df)
else:
    print("Populate CFG.data_dir with Ninapro DB2 .mat files, then re-run this cell.")


## 10. Leakage demonstration — the headline result

In [ ]:
if 'results_df' in dir() and len(results_df):
    summary = results_df.groupby(["protocol", "model"])["macro_f1"].mean().unstack("protocol")
    display(summary)

    fig, ax = plt.subplots(figsize=(7, 4))
    summary.plot(kind="bar", ax=ax)
    ax.set_ylabel("macro F1")
    ax.set_title("Within-subject vs LOSO vs (leaky) random-shuffle split")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig("project1_leakage_comparison.png", dpi=150)
    plt.show()

    gap = (summary["random_shuffle_leaky"] - summary["within_subject"]).dropna()
    print("Inflation from random-shuffle leakage (macro F1 points):")
    print((gap * 100).round(1))


## Notes / expected numbers

- Within-subject (protocol a): expect **75–90% macro F1** on 10 classes.
- LOSO (protocol b): expect a drop to **40–60% macro F1** — this is the honest cross-subject generalization number.
- Random shuffle (protocol c): expect this to **overstate accuracy by 20+ points** relative to (a), because overlapping 100 ms-hop windows from the same 5 s contraction land on both sides of the split. That gap, not the absolute accuracy, is the artifact worth screenshotting.
- The trained CNN and its z-score stats from the within-subject split are reused directly in `02_temporal_decision_aggregation.ipynb` and `03_quantization_efficiency.ipynb` — save `model.state_dict()` and `(mu, sd)` to disk (e.g. `torch.save`) once you're happy with a run.
